# Caso 1: SIMPLE - Calidad de Datos en Retail

---

## Contexto del Negocio

### Descripción del Problema
**MegaStore**, una tienda online de productos electrónicos, ha notado inconsistencias en sus reportes de ventas mensuales. Los gerentes reportan cifras diferentes dependiendo de qué sistema consulten, y el equipo de marketing no puede segmentar correctamente a sus clientes.

### Objetivo Analítico
Evaluar y mejorar la calidad de los datos transaccionales de ventas para:
- Generar reportes confiables de ingresos
- Segmentar clientes de manera efectiva
- Identificar productos con mejor desempeño

### Impacto de la Mala Calidad de Datos
- **Financiero**: Reportes de ingresos incorrectos pueden llevar a decisiones de inversión erróneas
- **Operativo**: Inventario mal calculado por datos duplicados o inconsistentes
- **Estratégico**: Campañas de marketing dirigidas a segmentos incorrectos

---

## Dimensiones de Calidad a Evaluar

En este caso trabajaremos con:

1. **Completitud**: ¿Tenemos todos los datos necesarios?
2. **Exactitud**: ¿Los valores son correctos?
3. **Consistencia**: ¿Los datos son coherentes entre sí?
4. **Integridad**: ¿Se mantienen las relaciones entre tablas?
5. **Razonabilidad**: ¿Los valores están dentro de rangos esperados?
6. **Oportunidad**: ¿Los datos están actualizados?
7. **Unicidad**: ¿Existen registros duplicados?
8. **Validez**: ¿Los formatos son correctos?

---

In [2]:
# Instalación de librerías necesarias
# !pip install pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline


In [3]:
df_avanzado = pd.read_parquet("dataset_calidad.parquet")

---

# SOLUCIÓN NIVEL AVANZADO

## Objetivo
Implementar un pipeline robusto y automatizado de calidad de datos:
- Validaciones basadas en reglas de negocio complejas
- Detección automática de anomalías
- Logging de problemas para auditoría
- Métricas de calidad cuantificables
- Proceso reproducible y escalable

In [4]:
quality_log = []

print(f"Registros iniciales: {len(df_avanzado)}\n")

Registros iniciales: 1000025



In [5]:
class DataQualityValidator:

    def __init__(self, df):
        self.df = df.copy()
        self.initial_count = len(df)
        self.now = datetime.now()
        self.quality_log = []  # 👈 AQUÍ ESTÁ LA CLAVE

    # LOGGING PROFESIONAL
    def log_issue(self, dimension, rule, affected_rows, action):
        self.quality_log.append({
            "timestamp": self.now,
            "dimension": dimension,
            "rule": rule,
            "affected_rows": int(affected_rows),
            "action": action
        })

    # 1. COMPLETITUD
    def validate_completeness(self):
        critical = ['id_transaccion', 'fecha', 'cliente_id']

        for col in critical:
            mask = self.df[col].isna()
            count = mask.sum()
            if count > 0:
                self.df = self.df.loc[~mask]
                self.log_issue("completitud", f"{col}_null", count, "drop")

        # imputación
        for col in ['metodo_pago', 'region']:
            mask = self.df[col].isna()
            count = mask.sum()
            if count > 0:
                fill = self.df[col].mode().iloc[0]
                self.df[col] = self.df[col].fillna(fill)
                self.log_issue("completitud", f"{col}_null", count, f"fill_mode:{fill}")

        # email
        mask = self.df['cliente_email'].isna()
        count = mask.sum()
        if count > 0:
            self.df.loc[mask, 'cliente_email'] = (
                self.df.loc[mask, 'cliente_id'].astype(str) + "@generated.com"
            )
            self.log_issue("completitud", "email_null", count, "generated")

    # 2. EXACTITUD
    def validate_accuracy(self):

        # precios <= 0
        mask = self.df['precio_unitario'] <= 0
        count = mask.sum()
        if count > 0:
            median = self.df.loc[~mask, 'precio_unitario'].median()
            self.df.loc[mask, 'precio_unitario'] = median
            self.log_issue("exactitud", "precio<=0", count, f"median:{median}")

        # descuentos
        mask = (self.df['descuento'] < 0) | (self.df['descuento'] > 1)
        count = mask.sum()
        if count > 0:
            self.df.loc[mask, 'descuento'] = 0.3
            self.log_issue("exactitud", "descuento_fuera_rango", count, "set_0.3")

        # cantidad
        mask = (self.df['cantidad'] <= 0) | (self.df['cantidad'] > 500)
        count = mask.sum()
        if count > 0:
            median = self.df.loc[~mask, 'cantidad'].median()
            self.df.loc[mask, 'cantidad'] = median
            self.log_issue("exactitud", "cantidad_invalida", count, f"median:{median}")

    # 3. CONSISTENCIA
    def validate_consistency(self):

        mapa = {
            'Laptop': 'Computadoras',
            'Monitor': 'Computadoras',
            'Mouse': 'Accesorios',
            'Teclado': 'Accesorios',
            'Audífonos': 'Accesorios',
            'Webcam': 'Accesorios',
            'SSD': 'Componentes',
            'RAM': 'Componentes'
        }

        expected = self.df['producto'].map(mapa)
        mask = self.df['categoria'] != expected
        count = mask.sum()

        if count > 0:
            self.df.loc[mask, 'categoria'] = expected[mask]
            self.log_issue("consistencia", "producto_categoria", count, "corrected")

        # monto
        calc = self.df['cantidad'] * self.df['precio_unitario'] * (1 - self.df['descuento'])
        mask = np.abs(self.df['monto_total'] - calc) > 0.01
        count = mask.sum()

        if count > 0:
            self.df.loc[mask, 'monto_total'] = calc[mask]
            self.log_issue("consistencia", "monto_incorrecto", count, "recalculated")

    # 4. INTEGRIDAD
    def validate_integrity(self):

        max_id = self.df['cliente_id'].quantile(0.99)

        mask = self.df['cliente_id'] > max_id * 2
        count = mask.sum()

        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("integridad", "cliente_id_outlier", count, "drop")

    # 5. RAZONABILIDAD
    def validate_reasonableness(self):

        q1 = self.df['precio_unitario'].quantile(0.25)
        q3 = self.df['precio_unitario'].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 3 * iqr
        upper = q3 + 3 * iqr

        mask = (self.df['precio_unitario'] < lower) | (self.df['precio_unitario'] > upper)
        count = mask.sum()

        if count > 0:
            self.df = self.df.loc[~mask]
            self.log_issue("razonabilidad", "precio_outlier", count, "drop")

    # 6. OPORTUNIDAD
    def validate_timeliness(self):

        now = datetime.now()
        mask_future = self.df['fecha'] > now
        count = mask_future.sum()

        if count > 0:
            self.df = self.df.loc[~mask_future]
            self.log_issue("oportunidad", "fecha_futura", count, "drop")

    # 7. UNICIDAD
    def validate_uniqueness(self):

        before = len(self.df)
        self.df = self.df.drop_duplicates()
        count = before - len(self.df)

        if count > 0:
            self.log_issue("unicidad", "duplicados", count, "drop")

    # 8. VALIDEZ
    def validate_validity(self):

        self.df['cliente_email'] = self.df['cliente_email'].str.lower().str.strip()

        mask = ~self.df['cliente_email'].str.contains("@", na=False)
        count = mask.sum()

        if count > 0:
            self.df.loc[mask, 'cliente_email'] = (
                self.df.loc[mask, 'cliente_id'].astype(str) + "@fix.com"
            )
            self.log_issue("validez", "email_invalido", count, "fixed")

    # PIPELINE
    def run(self):
        self.validate_completeness()
        self.validate_accuracy()
        self.validate_consistency()
        self.validate_integrity()
        self.validate_reasonableness()
        self.validate_timeliness()
        self.validate_uniqueness()
        self.validate_validity()
        return self.df

    # MÉTRICAS
    def report(self):
        final_count = len(self.df)

        print("\n==== REPORTE ====")
        print(f"Inicial: {self.initial_count}")
        print(f"Final: {final_count}")
        print(f"Loss %: {(1 - final_count/self.initial_count)*100:.2f}%")

        log_df = pd.DataFrame(self.quality_log)
        print("\nResumen por dimensión:")
        print(log_df.groupby("dimension")["affected_rows"].sum())

        return log_df


In [6]:
# Ejecutar validación avanzada
validator = DataQualityValidator(df_avanzado)
df_clean = validator.run()
log_df = validator.report()


==== REPORTE ====
Inicial: 1000025
Final: 1000017
Loss %: 0.00%

Resumen por dimensión:
dimension
completitud          30
consistencia     666567
exactitud            18
oportunidad           4
razonabilidad         4
validez               1
Name: affected_rows, dtype: int64


In [7]:
# Generar reporte final
log_df

,timestamp,dimension,rule,affected_rows,action
0,2026-04-21 15:15:59.899697,completitud,metodo_pago_null,10,fill_mode:Transferencia
1,2026-04-21 15:15:59.899697,completitud,region_null,10,fill_mode:Sur
2,2026-04-21 15:15:59.899697,completitud,email_null,10,generated
3,2026-04-21 15:15:59.899697,exactitud,precio<=0,8,median:754.43
4,2026-04-21 15:15:59.899697,exactitud,descuento_fuera_rango,5,set_0.3
5,2026-04-21 15:15:59.899697,exactitud,cantidad_invalida,5,median:2.0
6,2026-04-21 15:15:59.899697,consistencia,producto_categoria,666526,corrected
7,2026-04-21 15:15:59.899697,consistencia,monto_incorrecto,41,recalculated
8,2026-04-21 15:15:59.899697,razonabilidad,precio_outlier,4,drop
9,2026-04-21 15:15:59.899697,oportunidad,fecha_futura,4,drop


In [9]:
df_clean.to_parquet("avanzado")

### Conclusiones de la Solución Avanzada

**Ventajas:**
- Pipeline automatizado y reproducible
- Logging completo para auditoría
- Trata todas las dimensiones de calidad
- Aplicable a producción
- Trazabilidad de decisiones

**Características clave:**
- Modular y extensible
- Métricas cuantificables
- Reglas de negocio explícitas
- Fácil de mantener y actualizar